# Orchard ML: Edge Deployment with ONNX & Quantization

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tomrussobuilds/orchard-ml/blob/main/notebooks/03_edge_deploy_onnx_quantize.ipynb)

This notebook demonstrates Orchard ML's **production export pipeline** — from trained model to optimized edge-ready artifact:

- **Dataset**: [OrganCMNIST](https://medmnist.com/) (28×28 RGB, 11 organ classes)
- **Model**: MobileNetV3-Small via timm (~1.5M parameters, mobile-optimized)
- **Time**: ~15 minutes on Colab CPU

### What you'll learn
1. How `orchard run` trains, exports, quantizes, validates, and benchmarks in a **single command**
2. How to compare INT8 vs INT4 quantization using the Python API
3. How to evaluate the size–latency–accuracy trade-off for edge deployment

## 1. Setup

Clone the repository and install dependencies.

In [ ]:
import os

%cd /content
if not os.path.isdir("orchard-ml"):
    !git clone --depth 1 https://github.com/tomrussobuilds/orchard-ml.git

%cd /content/orchard-ml
!git fetch origin && git reset --hard origin/main
%pip install -q . --force-reinstall --no-deps

## 2. Configuration

We use **MobileNetV3-Small** — a 1.5M-parameter architecture designed for mobile and edge inference.
Pretrained ImageNet weights accelerate convergence even on a different domain (medical imaging),
so 10 epochs are enough to reach strong performance on OrganCMNIST.

The `export` section enables the full production pipeline:
- ONNX export with dynamic batch axes
- INT8 post-training quantization (qnnpack backend for ARM/mobile)
- Numerical validation (PyTorch vs ONNX vs quantized)
- Inference latency benchmark

In [ ]:
%%writefile colab_edge_deploy.yaml
# Edge Deployment Demo — MobileNetV3-Small on OrganCMNIST

dataset:
  name: "organcmnist"
  data_root: ./dataset
  resolution: 28
  lazy_loading: true
  force_rgb: true
  use_weighted_sampler: true

architecture:
  name: "timm/mobilenetv3_small_100"
  pretrained: true
  dropout: 0.2

training:
  seed: 42
  batch_size: 64
  learning_rate: 0.0005
  weight_decay: 1e-4
  momentum: 0.9
  min_lr: 1e-6
  mixup_alpha: 0.1
  label_smoothing: 0.1
  epochs: 10
  patience: 8
  grad_clip: 1.0
  mixup_epochs: 0
  monitor_metric: "auc"
  scheduler_type: "cosine"
  use_amp: false
  use_tta: false
  criterion_type: "cross_entropy"

augmentation:
  hflip: 0.5
  rotation_angle: 10
  jitter_val: 0.2
  min_scale: 0.9

hardware:
  device: "auto"

telemetry:
  output_dir: ./outputs
  log_level: "INFO"
  log_interval: 50

evaluation:
  n_samples: 16
  fig_dpi: 200
  cmap_confusion: Blues
  plot_style: seaborn-v0_8-muted
  grid_cols: 4
  fig_size_predictions: [12, 8]
  report_format: xlsx
  save_confusion_matrix: true
  save_predictions_grid: true

tracking:
  enabled: false

export:
  format: onnx
  opset_version: 18
  quantize: true
  quantization_type: int8
  quantization_backend: qnnpack
  validate_export: true
  benchmark: true
  benchmark_runs: 50

## 3. Train + Export

A single `orchard run` command executes the full pipeline:

1. **Download** OrganCMNIST (automatic, cached in `./dataset`)
2. **Train** MobileNetV3-Small for 10 epochs with cosine scheduling
3. **Evaluate** — confusion matrix, sample predictions, classification report
4. **Export** to ONNX with dynamic batch axes
5. **Quantize** to INT8 (qnnpack backend)
6. **Validate** — numerical comparison PyTorch vs ONNX vs quantized
7. **Benchmark** — inference latency (FP32 vs INT8)

In [ ]:
!orchard run colab_edge_deploy.yaml

## 4. Explore Artifacts

Orchard ML saves all outputs to a timestamped directory under `outputs/`.

In [ ]:
import glob
import os

# Find the latest run directory
run_dirs = sorted(glob.glob("outputs/*/"))
latest_run = run_dirs[-1]
print(f"Run directory: {latest_run}\n")

# List all generated artifacts
for root, dirs, files in os.walk(latest_run):
    level = root.replace(latest_run, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    sub_indent = "  " * (level + 1)
    for file in sorted(files):
        size = os.path.getsize(os.path.join(root, file))
        print(f"{sub_indent}{file} ({size / 1024:.1f} KB)")

In [ ]:
from IPython.display import display, Image

# Confusion matrix
cm_files = glob.glob(f"{latest_run}/figures/confusion_matrix*.png")
if cm_files:
    print("Confusion Matrix:")
    display(Image(filename=cm_files[0], width=600))

In [ ]:
# Sample predictions
pred_files = glob.glob(f"{latest_run}/figures/sample_predictions*.png")
if pred_files:
    print("Sample Predictions:")
    display(Image(filename=pred_files[0], width=700))

## 5. INT4 Quantization via Python API

The CLI produced an INT8 model. Now we use the Python API directly to create an **INT4 variant** and compare.

INT4 quantization targets only `Gemm` nodes (fully-connected layers). Conv layers stay at FP32,
and `MatMul` nodes (attention layers in transformers) require 8-bit — so the actual compression
depends on the architecture. MobileNetV3 is mostly convolutions, so INT4 savings are modest here.

In [ ]:
from pathlib import Path

from orchard.export import quantize_model, validate_export, benchmark_onnx_inference

# Locate the exported ONNX model
onnx_files = glob.glob(f"{latest_run}/exports/model*.onnx")
# Pick the non-quantized model (no 'quantized' in name)
onnx_fp32 = Path([f for f in onnx_files if "quantized" not in f][0])
onnx_int8 = Path([f for f in onnx_files if "quantized" in f][0])
print(f"FP32 model: {onnx_fp32}")
print(f"INT8 model: {onnx_int8}")

In [ ]:
# Quantize to INT4
onnx_int4 = quantize_model(
    onnx_path=onnx_fp32,
    output_path=onnx_fp32.parent / "model_quantized_int4.onnx",
    backend="qnnpack",
    weight_type="int4",
)
print(f"INT4 model: {onnx_int4}")

## 6. Numerical Validation

Compare PyTorch outputs against each ONNX variant. The validation runs random inputs through
both runtimes and checks that the maximum absolute deviation stays below a threshold.

- **FP32 ONNX**: should match PyTorch within ~1e-5 (floating-point rounding only)
- **INT8**: slightly higher deviation from weight quantization
- **INT4**: highest deviation, but still acceptable for classification tasks

In [ ]:
import torch
from orchard.architectures import build_model
from orchard.core.config import Config

# Rebuild the model and load trained weights for validation
checkpoint_files = glob.glob(f"{latest_run}/checkpoints/best_model*.pth")
checkpoint_path = Path(checkpoint_files[0])

# Load the saved config to rebuild the exact model
import yaml

config_files = glob.glob(f"{latest_run}/reports/config*.yaml")
with open(config_files[0]) as f:
    saved_cfg = yaml.safe_load(f)

num_classes = saved_cfg["dataset"]["num_classes"]
in_channels = saved_cfg["dataset"]["in_channels"]

model = build_model(
    arch_name=saved_cfg["architecture"]["name"],
    num_classes=num_classes,
    in_channels=in_channels,
    pretrained=False,  # We load from checkpoint
    dropout=saved_cfg["architecture"]["dropout"],
)

checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    model.load_state_dict(checkpoint["model_state_dict"])
else:
    model.load_state_dict(checkpoint)
model.eval()

input_shape = (in_channels, 28, 28)
print(f"Model loaded: {sum(p.numel() for p in model.parameters()):,} parameters")
print(f"Input shape: {input_shape}")
print(f"Classes: {num_classes}")

In [ ]:
# Validate all three variants
for label, path in [("FP32 ONNX", onnx_fp32), ("INT8", onnx_int8), ("INT4", onnx_int4)]:
    if path and path.exists():
        validate_export(
            pytorch_model=model,
            onnx_path=path,
            input_shape=input_shape,
            num_samples=20,
            max_deviation=0.1,  # Relaxed threshold for quantized models
            label=label,
        )

## 7. Latency Benchmark

Measure average inference time (single sample, CPU) across all three variants.
Quantized models should show lower latency thanks to integer arithmetic.

In [ ]:
results = {}

for label, path in [("FP32", onnx_fp32), ("INT8", onnx_int8), ("INT4", onnx_int4)]:
    if path and path.exists():
        latency = benchmark_onnx_inference(
            onnx_path=path,
            input_shape=input_shape,
            num_runs=100,
            seed=42,
            label=label,
        )
        size_mb = path.stat().st_size / (1024 * 1024)
        results[label] = {"latency_ms": latency, "size_mb": size_mb}

## 8. Summary

Side-by-side comparison of all exported variants.

In [ ]:
print(f"{'Variant':<10} {'Size (MB)':>10} {'Latency (ms)':>14} {'Compression':>13}")
print("-" * 50)

fp32_size = results.get("FP32", {}).get("size_mb", 1)

for label, data in results.items():
    ratio = fp32_size / data["size_mb"] if data["size_mb"] > 0 else 0
    print(
        f"{label:<10} {data['size_mb']:>9.2f}  {data['latency_ms']:>13.2f}  {ratio:>12.1f}x"
    )

## 9. Next Steps

- **Different quantization**: Try `uint8` (unsigned) or switch backend to `fbgemm` for x86 servers
- **Higher resolution**: Use `config_timm_efficientnet_lite0_128.yaml` for 128×128 with GPU
- **Model search**: See [02_galaxy10_optuna_model_search.ipynb](./02_galaxy10_optuna_model_search.ipynb) for Optuna-powered architecture selection
- **Deploy**: The quantized `.onnx` file runs anywhere via [ONNX Runtime](https://onnxruntime.ai/) — Python, C++, mobile (Android/iOS), or edge devices (Raspberry Pi, Jetson)
- **Full recipe catalog**: Browse all pre-configured recipes in the [recipes/](https://github.com/tomrussobuilds/orchard-ml/tree/main/recipes) directory